In [7]:
import pandas as pd
import numpy as np
import re

# Paths
MAPPING = "../Progress/code/datasets/mapping"
OG      = "../Progress/code/datasets/og"

# Load
occ_base     = pd.read_csv(f"{MAPPING}/occupation_base_soc.csv")
skill_bridge = pd.read_csv(f"{MAPPING}/occupation_skill_bridge_onet.csv", low_memory=False)
tech_summary = pd.read_csv(f"{MAPPING}/occupation_technology_skills_onet_summary.csv")
best_match   = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_best_thr30.csv")
salary       = pd.read_excel(f"{OG}/occupation-salary.xlsx")
automation   = pd.read_csv(f"{OG}/automation-data-by-state.csv", encoding="latin-1")

print("All files loaded.")
print(f"  occ_base:     {occ_base.shape}")
print(f"  skill_bridge: {skill_bridge.shape}")
print(f"  tech_summary: {tech_summary.shape}")
print(f"  best_match:   {best_match.shape}")
print(f"  salary:       {salary.shape}")
print(f"  automation:   {automation.shape}")

All files loaded.
  occ_base:     (702, 4)
  skill_bridge: (94682, 13)
  tech_summary: (618, 6)
  best_match:   (57085, 8)
  salary:       (1394, 20)
  automation:   (702, 54)


In [8]:
# SOC code aliases — occupation_base uses old codes, salary file uses new ones
soc_alias = {
    "29-1111": "29-1141",   # Registered Nurses
    "29-1151": "29-1151",   # Nurse Anesthetists (same)
    "29-1161": "29-1161",   # Nurse Midwives (same)
    "29-1171": "29-1171",   # Nurse Practitioners (same)
}

occ_base["SOC6"]        = occ_base["SOC6"].astype(str).str.strip()
occ_base["SOC6_salary"] = occ_base["SOC6"].replace(soc_alias)

salary_detail = salary[salary["OCC_GROUP"] == "detailed"].copy()
salary_detail["SOC6"] = salary_detail["OCC_CODE"].astype(str).str.strip()

salary_cols = salary_detail[["SOC6", "TOT_EMP", "A_MEDIAN", "H_PCT10", "H_PCT90"]].copy()
salary_cols.replace("*", np.nan, inplace=True)
for col in ["TOT_EMP", "A_MEDIAN", "H_PCT10", "H_PCT90"]:
    salary_cols[col] = pd.to_numeric(salary_cols[col], errors="coerce")

occ_enriched = occ_base.merge(
    salary_cols,
    left_on="SOC6_salary",
    right_on="SOC6",
    how="left",
    suffixes=("", "_sal")
)
occ_enriched.drop(columns=["SOC6_sal", "SOC6_salary"], inplace=True)

print(f"Enriched: {occ_enriched.shape}")
print(f"With salary: {occ_enriched['A_MEDIAN'].notna().sum()}")
print(f"Missing salary: {occ_enriched['A_MEDIAN'].isna().sum()}")
print("\nNursing check:")
print(occ_enriched[occ_enriched["occupation_name"].str.contains("Nurse|nurse", na=False)]
      [["SOC6", "occupation_name", "A_MEDIAN", "TOT_EMP"]])

Enriched: (702, 8)
With salary: 680
Missing salary: 22

Nursing check:
        SOC6                                    occupation_name  A_MEDIAN  \
242  29-1111                                  Registered Nurses   68450.0   
266  29-2061  Licensed Practical and Licensed Vocational Nurses   44090.0   

       TOT_EMP  
242  2857180.0  
266   702400.0  


In [9]:
df = occ_enriched.copy()

# Wage stability — lower spread = more stable, so invert
df["wage_spread"] = (df["H_PCT90"] - df["H_PCT10"]).clip(lower=0)

def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

df["income_strength"]   = minmax(df["A_MEDIAN"].fillna(df["A_MEDIAN"].median()))
df["employment_scale"]  = minmax(df["TOT_EMP"].fillna(df["TOT_EMP"].median()))
df["wage_stability"]    = 1 - minmax(df["wage_spread"].fillna(df["wage_spread"].median()))
df["automation_safety"] = 1 - df["automation_probability"].fillna(df["automation_probability"].median())

df["GCSI"] = (
    0.35 * df["income_strength"] +
    0.25 * df["employment_scale"] +
    0.20 * df["wage_stability"]  +
    0.20 * df["automation_safety"]
) * 100
df["GCSI"] = df["GCSI"].round(2)

occupation_gcsi = df[[
    "SOC6", "occupation_name", "onet_title",
    "automation_probability", "A_MEDIAN", "TOT_EMP",
    "income_strength", "employment_scale",
    "wage_stability", "automation_safety", "GCSI"
]].copy()

occupation_gcsi.to_csv(f"{MAPPING}/occupation_gcsi.csv", index=False)

print(f"occupation_gcsi.csv saved: {occupation_gcsi.shape}")
print("\nTop 10:")
print(occupation_gcsi[["occupation_name", "GCSI", "A_MEDIAN"]]
      .sort_values("GCSI", ascending=False).head(10).to_string())
print("\nBottom 10:")
print(occupation_gcsi[["occupation_name", "GCSI"]]
      .sort_values("GCSI").head(10).to_string())

occupation_gcsi.csv saved: (702, 11)

Top 10:
                               occupation_name   GCSI  A_MEDIAN
0                             Chief Executives  71.41  181210.0
232                          Dentists; General  65.07  153900.0
7    Computer and Information Systems Managers  61.91  135800.0
1              General and Operations Managers  61.66   99310.0
3                           Marketing Managers  60.52  131180.0
242                          Registered Nurses  60.24   68450.0
165                                    Lawyers  59.57  118160.0
8                           Financial Managers  59.24  121750.0
4                               Sales Managers  58.56  117960.0
241                                Podiatrists  58.25  124830.0

Bottom 10:
                                   occupation_name   GCSI
377                            Real Estate Brokers  12.65
39                          Farm Labor Contractors  15.37
210  Umpires; Referees; and Other Sports Officials  17.36
376   

In [10]:
# Name resolution map — broad/grouped BLS names → specific GCSI occupation names
full_map = {
    "biological scientists":                              "biological scientists; all other",
    "psychologists":                                      "clinical; counseling; and school psychologists",
    "education administrators":                           "education administrators; all other",
    "education administrators postsecondary":             "education administrators; postsecondary",
    "foreign language and literature teachers postsecondary": "postsecondary teachers; all other",
    "legal occupations":                                  "lawyers",
    "english language and literature teachers postsecondary": "postsecondary teachers; all other",
    "engineers":                                          "engineers; all other",
    "media and communication workers":                    "public relations specialists",
    "media and communication workers all other":          "public relations specialists",
    "business operations specialists":                    "business operations specialists; all other",
    "business operations specialists all other":          "business operations specialists; all other",
    "community health workers":                           "health educators",
    "management occupations":                             "general and operations managers",
    "medical scientists":                                 "medical scientists; except epidemiologists",
    "art drama and music teachers postsecondary":         "postsecondary teachers; all other",
    "social workers":                                     "social workers; all other",
    "electrical and electronics engineers":               "electrical engineers",
    "philosophy and religion teachers postsecondary":     "postsecondary teachers; all other",
    "writers and editors":                                "writers and authors",
    "criminal justice and law enforcement teachers postsecondary": "postsecondary teachers; all other",
    "environmental scientists and geoscientists":         "environmental scientists and specialists; including health",
    "clinical counseling and school psychologists":       "clinical; counseling; and school psychologists",
    "special education teachers":                         "special education teachers; all other",
    "special education teachers secondary school":        "special education teachers; secondary school",
    "exercise physiologists":                             "physical therapists",
    "area ethnic and cultural studies teachers postsecondary": "postsecondary teachers; all other",
    "information security analysts":                      "information security analysts",
    "architecture and engineering occupations":           "architects; except landscape and naval",
    "software developers systems software":               "software developers",
    "art and design workers":                             "art directors",
    "fine artists including painters sculptors and illustrators": "fine artists; including painters; sculptors; and illustrators",
    "nurse practitioners":                                "nurse practitioners",
    "nurse anesthetists":                                 "nurse anesthetists",
    "nurse midwives":                                     "nurse midwives",
    "nurse midwives and nurse anesthetists":              "nurse anesthetists",
    "designers":                                          "graphic designers",
    "chemists and materials scientists":                  "chemists",
    "other teachers and instructors":                     "postsecondary teachers; all other",
    "elementary and middle school teachers":              "elementary school teachers; except special education",
    "financial specialists":                              "financial analysts",
    "counselors all other":                               "counselors; all other",
    "conservation scientists and foresters":              "conservation scientists",
    "religious workers":                                  "clergy",
    "communications equipment operators":                 "telephone operators",
    "mathematical science teachers postsecondary":        "postsecondary teachers; all other",
    "computer network architects":                        "computer network architects",
    "biological science teachers postsecondary":          "postsecondary teachers; all other",
    "genetic counselors":                                 "genetic counselors",
    "actors producers and directors":                     "producers and directors",
    "agricultural and food scientists":                   "food scientists and technologists",
    "agents and business managers of artists performers and athletes": "agents and business managers of artists; performers; and athletes",
    "administrative law judges adjudicators and hearing officers": "administrative law judges; adjudicators; and hearing officers",
    "advertising marketing promotions public relations and sales managers": "marketing managers",
    "agricultural equipment operators":                   "agricultural workers; all other",
    "agricultural workers":                               "agricultural workers; all other",
    "agricultural workers all other":                     "agricultural workers; all other",
    "air traffic controllers and airfield operations specialists": "air traffic controllers",
    "air transportation workers":                         "airline pilots; copilots; and flight engineers",
    "aircraft pilots and flight engineers":               "airline pilots; copilots; and flight engineers",
    "aircraft structure surfaces rigging and systems assemblers": "aircraft structure; surfaces; rigging; and systems assemblers",
    "anesthesiologists":                                  "anesthesiologists and anesthesiologist assistants",
    "animal care and service workers":                    "animal care and service workers; all other",
    "arbitrators mediators and conciliators":             "arbitrators; mediators; and conciliators",
    "architects except landscape and naval":              "architects; except landscape and naval",
    "architects except naval":                            "architects; except landscape and naval",
    "architects surveyors and cartographers":             "architects; except landscape and naval",
    "preschool and kindergarten teachers":                "preschool teachers; except special education",
    "psychology teachers postsecondary":                  "postsecondary teachers; all other",
    "sales and related occupations":                      "sales representatives; wholesale and manufacturing",
    "secondary school teachers":                          "secondary school teachers; except special and career/technical education",
    "arts design entertainment sports and media occupations": "art directors",
    "personal care and service occupations":              "personal care aides",
    "physical scientists":                                "physical scientists; all other",
}

def resolve(name):
    if pd.isna(name):
        return name
    return full_map.get(str(name).lower().strip(), str(name).lower().strip())

# Apply map
bm = best_match.copy()
bm["career_key"] = bm["career_match"].apply(resolve)

# Build lookup from GCSI table
gcsi_lookup = occupation_gcsi.copy()
gcsi_lookup["lookup_key"] = gcsi_lookup["occupation_name"].str.lower().str.strip()

# Join
best_enriched = bm.merge(
    gcsi_lookup[["lookup_key", "SOC6", "GCSI",
                 "automation_probability", "A_MEDIAN", "TOT_EMP"]],
    left_on="career_key",
    right_on="lookup_key",
    how="left"
)

# Fill unmatched GCSI with median
gcsi_median = best_enriched["GCSI"].median()
best_enriched["GCSI_imputed"] = best_enriched["GCSI"].isna()
best_enriched["GCSI"]         = best_enriched["GCSI"].fillna(gcsi_median)

# # Hybrid score
# best_enriched["hybrid_score"] = (
#     0.50 * best_enriched["similarity"] +
#     0.50 * (best_enriched["GCSI"] / 100)
# ).round(4)

# Final hybrid score — 40/60 with sim-scaled GCSI (threshold 0.65)
best_enriched["hybrid_score"] = (
    0.40 * best_enriched["similarity"] +
    0.60 * (best_enriched["GCSI"] / 100) * (best_enriched["similarity"] / 0.65).clip(upper=1.0)
).round(4)

best_enriched.drop(columns=["lookup_key", "career_key"], inplace=True)
best_enriched["SOC6"] = best_enriched["SOC6"].astype(str).str.strip()
best_enriched.to_csv(f"{MAPPING}/program_best_enriched_gcsi.csv", index=False)

covered = best_enriched[~best_enriched["GCSI_imputed"]]["program_name"].nunique()
total   = best_enriched["program_name"].nunique()

print(f"Saved: {best_enriched.shape}")
print(f"Real GCSI rows:    {(~best_enriched['GCSI_imputed']).sum()}")
print(f"Imputed GCSI rows: {best_enriched['GCSI_imputed'].sum()}")
print(f"Program coverage:  {covered}/{total} = {covered/total*100:.1f}%")
print(f"GCSI median used for imputation: {gcsi_median:.2f}")

Saved: (57085, 15)
Real GCSI rows:    45120
Imputed GCSI rows: 11965
Program coverage:  17259/23780 = 72.6%
GCSI median used for imputation: 41.78


In [11]:
# Top 5 skills per occupation by importance
top_skills = (
    skill_bridge
    .dropna(subset=["SOC6", "skill_name", "importance"])
    .sort_values("importance", ascending=False)
    .groupby("SOC6")
    .head(5)
    .groupby("SOC6")["skill_name"]
    .apply(list)
    .reset_index()
    .rename(columns={"skill_name": "top_skills"})
)
top_skills["SOC6"] = top_skills["SOC6"].astype(str).str.strip()

# Tech summary
top_tech = tech_summary[["SOC6", "technology_items",
                          "hot_tech_items", "in_demand_items"]].copy()
top_tech["SOC6"] = top_tech["SOC6"].astype(str).str.strip()

print(f"top_skills: {top_skills.shape}")
print(f"top_tech:   {top_tech.shape}")

top_skills: (617, 2)
top_tech:   (618, 4)


In [12]:
def recommend(degree_name, top_n=10):
    """
    Input:  degree name string (partial match, case insensitive)
    Output: DataFrame of top N careers ranked by hybrid score
            with GCSI, salary, skills, and tech tools
    """
    mask = best_enriched["degree_final"].str.lower().str.contains(
        degree_name.lower().strip(), na=False
    )
    results = best_enriched[mask].copy()

    if results.empty:
        print(f"No matches found for: '{degree_name}'")
        return pd.DataFrame()

    results = (
        results
        .sort_values("hybrid_score", ascending=False)
        .drop_duplicates(subset=["career_match"])
        .head(top_n)
    )

    results = results.merge(top_skills, on="SOC6", how="left")
    results = results.merge(top_tech,   on="SOC6", how="left")

    return results[[
        "program_name", "program_type", "career_match",
        "similarity", "GCSI", "GCSI_imputed", "hybrid_score",
        "A_MEDIAN", "automation_probability",
        "top_skills", "technology_items", "hot_tech_items"
    ]].reset_index(drop=True)


# Quick tests
for degree in ["computer science", "economics", "business administration",
               "nursing", "public administration"]:
    print(f"\n=== {degree} ===")
    r = recommend(degree, top_n=5)
    print(r[["career_match", "GCSI", "hybrid_score", "A_MEDIAN"]].to_string())


=== computer science ===


                                   career_match   GCSI  hybrid_score  A_MEDIAN
0     computer and information systems managers  61.91        0.6477  135800.0
1          electrical and electronics engineers  45.03        0.6110   94210.0
2  computer and information research scientists  47.74        0.5942  111840.0
3                   computer hardware engineers  44.54        0.5863  115080.0
4                     computer systems analysts  48.11        0.5797   87220.0

=== economics ===
                                career_match   GCSI  hybrid_score  A_MEDIAN
0                         financial managers  59.24        0.6615  121750.0
1  computer and information systems managers  61.91        0.6365  135800.0
2                         marketing managers  60.52        0.6150  131180.0
3           economics teachers postsecondary  41.78        0.5747       NaN
4                   human resources managers  44.20        0.5731  106910.0

=== business administration ===
                  

In [13]:
import pandas as pd
import numpy as np
import re

MAPPING = "../Progress/code/datasets/mapping"
OG      = "../Progress/code/datasets/og"

# Load candidates (all careers per program)
print("Loading candidates table (4.3M rows)...")
candidates = pd.read_csv(
    f"{MAPPING}/program_career_sentence_transformer_candidates_thr30_long.csv"
)
print(f"Loaded: {candidates.shape}")

# Load GCSI table
occupation_gcsi = pd.read_csv(f"{MAPPING}/occupation_gcsi.csv")

# Load field mapping
masters_fields = pd.read_csv(
    "../Progress/code/datasets/cleaned/masters_with_fields.csv"
)[["program_name", "program_type", "broad_field"]].drop_duplicates()

print(f"occupation_gcsi: {occupation_gcsi.shape}")
print(f"masters_fields:  {masters_fields.shape}")

Loading candidates table (4.3M rows)...
Loaded: (4357770, 8)
occupation_gcsi: (702, 11)
masters_fields:  (27793, 3)


In [14]:
# ── Step 1: Apply name resolution map to candidates ──────────────────────────
full_map = {
    "biological scientists":                              "biological scientists; all other",
    "psychologists":                                      "clinical; counseling; and school psychologists",
    "education administrators":                           "education administrators; all other",
    "education administrators postsecondary":             "education administrators; postsecondary",
    "foreign language and literature teachers postsecondary": "postsecondary teachers; all other",
    "legal occupations":                                  "lawyers",
    "english language and literature teachers postsecondary": "postsecondary teachers; all other",
    "engineers":                                          "engineers; all other",
    "media and communication workers":                    "public relations specialists",
    "media and communication workers all other":          "public relations specialists",
    "business operations specialists":                    "business operations specialists; all other",
    "business operations specialists all other":          "business operations specialists; all other",
    "community health workers":                           "health educators",
    "management occupations":                             "general and operations managers",
    "medical scientists":                                 "medical scientists; except epidemiologists",
    "art drama and music teachers postsecondary":         "postsecondary teachers; all other",
    "social workers":                                     "social workers; all other",
    "electrical and electronics engineers":               "electrical engineers",
    "philosophy and religion teachers postsecondary":     "postsecondary teachers; all other",
    "writers and editors":                                "writers and authors",
    "criminal justice and law enforcement teachers postsecondary": "postsecondary teachers; all other",
    "environmental scientists and geoscientists":         "environmental scientists and specialists; including health",
    "clinical counseling and school psychologists":       "clinical; counseling; and school psychologists",
    "special education teachers":                         "special education teachers; all other",
    "special education teachers secondary school":        "special education teachers; secondary school",
    "exercise physiologists":                             "physical therapists",
    "area ethnic and cultural studies teachers postsecondary": "postsecondary teachers; all other",
    "information security analysts":                      "information security analysts",
    "architecture and engineering occupations":           "architects; except landscape and naval",
    "software developers systems software":               "software developers",
    "art and design workers":                             "art directors",
    "fine artists including painters sculptors and illustrators": "fine artists; including painters; sculptors; and illustrators",
    "nurse practitioners":                                "nurse practitioners",
    "nurse anesthetists":                                 "nurse anesthetists",
    "nurse midwives":                                     "nurse midwives",
    "nurse midwives and nurse anesthetists":              "nurse anesthetists",
    "designers":                                          "graphic designers",
    "chemists and materials scientists":                  "chemists",
    "other teachers and instructors":                     "postsecondary teachers; all other",
    "elementary and middle school teachers":              "elementary school teachers; except special education",
    "financial specialists":                              "financial analysts",
    "counselors all other":                               "counselors; all other",
    "conservation scientists and foresters":              "conservation scientists",
    "religious workers":                                  "clergy",
    "communications equipment operators":                 "telephone operators",
    "mathematical science teachers postsecondary":        "postsecondary teachers; all other",
    "computer network architects":                        "computer network architects",
    "biological science teachers postsecondary":          "postsecondary teachers; all other",
    "genetic counselors":                                 "genetic counselors",
    "actors producers and directors":                     "producers and directors",
    "agricultural and food scientists":                   "food scientists and technologists",
    "agents and business managers of artists performers and athletes": "agents and business managers of artists; performers; and athletes",
    "administrative law judges adjudicators and hearing officers": "administrative law judges; adjudicators; and hearing officers",
    "advertising marketing promotions public relations and sales managers": "marketing managers",
    "agricultural equipment operators":                   "agricultural workers; all other",
    "agricultural workers":                               "agricultural workers; all other",
    "agricultural workers all other":                     "agricultural workers; all other",
    "air traffic controllers and airfield operations specialists": "air traffic controllers",
    "air transportation workers":                         "airline pilots; copilots; and flight engineers",
    "aircraft pilots and flight engineers":               "airline pilots; copilots; and flight engineers",
    "aircraft structure surfaces rigging and systems assemblers": "aircraft structure; surfaces; rigging; and systems assemblers",
    "anesthesiologists":                                  "anesthesiologists and anesthesiologist assistants",
    "animal care and service workers":                    "animal care and service workers; all other",
    "arbitrators mediators and conciliators":             "arbitrators; mediators; and conciliators",
    "architects except landscape and naval":              "architects; except landscape and naval",
    "architects except naval":                            "architects; except landscape and naval",
    "architects surveyors and cartographers":             "architects; except landscape and naval",
    "preschool and kindergarten teachers":                "preschool teachers; except special education",
    "psychology teachers postsecondary":                  "postsecondary teachers; all other",
    "sales and related occupations":                      "sales representatives; wholesale and manufacturing",
    "secondary school teachers":                          "secondary school teachers; except special and career/technical education",
    "arts design entertainment sports and media occupations": "art directors",
    "personal care and service occupations":              "personal care aides",
    "physical scientists":                                "physical scientists; all other",
}

def resolve(name):
    if pd.isna(name): return name
    return full_map.get(str(name).lower().strip(), str(name).lower().strip())

print("Applying name resolution map...")
candidates["career_key"] = candidates["career_match"].apply(resolve)

# ── Step 2: Join GCSI onto candidates ────────────────────────────────────────
gcsi_lookup = occupation_gcsi.copy()
gcsi_lookup["lookup_key"] = gcsi_lookup["occupation_name"].str.lower().str.strip()

print("Joining GCSI scores...")
enriched = candidates.merge(
    gcsi_lookup[["lookup_key", "SOC6", "GCSI",
                 "automation_probability", "A_MEDIAN", "TOT_EMP",
                 "income_strength", "employment_scale",
                 "wage_stability", "automation_safety"]],
    left_on="career_key",
    right_on="lookup_key",
    how="left"
)

# Fill missing GCSI with median
gcsi_median = enriched["GCSI"].median()
enriched["GCSI_imputed"] = enriched["GCSI"].isna()
enriched["GCSI"]         = enriched["GCSI"].fillna(gcsi_median)
enriched["SOC6"]         = enriched["SOC6"].astype(str).str.strip()

# ── Step 3: Compute hybrid score ──────────────────────────────────────────────
print("Computing hybrid scores...")
enriched["hybrid_score"] = (
    0.40 * enriched["similarity"] +
    0.60 * (enriched["GCSI"] / 100) *
    (enriched["similarity"] / 0.65).clip(upper=1.0)
).round(4)

# ── Step 4: Keep top 15 per program (deduplicated by career) ─────────────────
print("Ranking and keeping top 15 per program...")
top15 = (
    enriched
    .sort_values("hybrid_score", ascending=False)
    .drop_duplicates(subset=["program_name", "program_type", "career_match"])
    .groupby(["program_name", "program_type"])
    .head(15)
    .reset_index(drop=True)
)

# ── Step 5: Attach broad field ───────────────────────────────────────────────
print("Attaching broad fields...")
top15 = top15.merge(
    masters_fields,
    on=["program_name", "program_type"],
    how="left"
)

# ── Step 6: Add candidate count per program ───────────────────────────────────
career_counts = (
    enriched
    .drop_duplicates(subset=["program_name", "program_type", "career_match"])
    .groupby(["program_name", "program_type"])
    .size()
    .reset_index(name="total_unique_careers")
)

top15 = top15.merge(career_counts, on=["program_name", "program_type"], how="left")

# ── Step 7: Save ─────────────────────────────────────────────────────────────
out_path = f"{MAPPING}/program_recommendations.csv"
top15.to_csv(out_path, index=False)

print(f"\nSaved: program_recommendations.csv")
print(f"Shape: {top15.shape}")
print(f"Unique programs: {top15['program_name'].nunique()}")
print(f"Avg careers per program: {top15.groupby(['program_name','program_type']).size().mean():.1f}")
print(f"GCSI coverage: {(~top15['GCSI_imputed']).sum()} real, {top15['GCSI_imputed'].sum()} imputed")

print("\n=== Sample: Economics MSc top 10 ===")
econ = top15[
    (top15["program_name"] == "Economics") &
    (top15["program_type"] == "MSc")
][["career_match", "similarity", "GCSI", "hybrid_score", "A_MEDIAN"]].head(10)
print(econ.to_string())

Applying name resolution map...
Joining GCSI scores...
Computing hybrid scores...
Ranking and keeping top 15 per program...
Attaching broad fields...

Saved: program_recommendations.csv
Shape: (365035, 23)
Unique programs: 23780
Avg careers per program: 13.1
GCSI coverage: 208328 real, 156707 imputed

=== Sample: Economics MSc top 10 ===
                                 career_match  similarity   GCSI  hybrid_score  A_MEDIAN
5955                               economists      0.8615  34.40        0.5510  101050.0
85330        economics teachers postsecondary      0.5289  39.90        0.4064       NaN
95357                      financial managers      0.4207  59.24        0.3983  121750.0
107417                         mathematicians      0.4742  45.57        0.3892  105810.0
114958                           sociologists      0.5004  39.76        0.3838   79750.0
115762                     marketing managers      0.3998  60.52        0.3833  131180.0
150839                     financial 

In [15]:
# Check GCSI for economists — why is it 34.40?
print("Economists in GCSI table:")
print(occupation_gcsi[
    occupation_gcsi["occupation_name"].str.lower().str.contains("economist", na=False)
][["SOC6", "occupation_name", "GCSI", "A_MEDIAN", "TOT_EMP"]].to_string())

# Check what career_key economists resolves to
print("\nHow 'economists' resolves:")
test = "economists"
print(f"  '{test}' → '{resolve(test)}'")

# Check hybrid score math manually
sim  = 0.8615
gcsi = 34.40
penalty = min(1.0, sim / 0.65)
hybrid  = 0.40 * sim + 0.60 * (gcsi / 100) * penalty
print(f"\nManual check economists:")
print(f"  sim={sim}, gcsi={gcsi}, penalty={penalty:.4f}")
print(f"  hybrid = 0.40*{sim} + 0.60*{gcsi/100:.4f}*{penalty:.4f} = {hybrid:.4f}")

sim2  = 0.4207
gcsi2 = 59.24
penalty2 = min(1.0, sim2 / 0.65)
hybrid2  = 0.40 * sim2 + 0.60 * (gcsi2 / 100) * penalty2
print(f"\nManual check financial managers:")
print(f"  sim={sim2}, gcsi={gcsi2}, penalty={penalty2:.4f}")
print(f"  hybrid = 0.40*{sim2} + 0.60*{gcsi2/100:.4f}*{penalty2:.4f} = {hybrid2:.4f}")

# What does financial managers score if similarity was higher?
print("\nFinancial managers at different similarities:")
for s in [0.42, 0.50, 0.60, 0.70, 0.76]:
    p = min(1.0, s / 0.65)
    h = 0.40 * s + 0.60 * (59.24 / 100) * p
    print(f"  sim={s:.2f} → hybrid={h:.4f}")

Economists in GCSI table:
        SOC6 occupation_name  GCSI  A_MEDIAN  TOT_EMP
130  19-3011      Economists  34.4  101050.0  19380.0

How 'economists' resolves:
  'economists' → 'economists'

Manual check economists:
  sim=0.8615, gcsi=34.4, penalty=1.0000
  hybrid = 0.40*0.8615 + 0.60*0.3440*1.0000 = 0.5510

Manual check financial managers:
  sim=0.4207, gcsi=59.24, penalty=0.6472
  hybrid = 0.40*0.4207 + 0.60*0.5924*0.6472 = 0.3983

Financial managers at different similarities:
  sim=0.42 → hybrid=0.3977
  sim=0.50 → hybrid=0.4734
  sim=0.60 → hybrid=0.5681
  sim=0.70 → hybrid=0.6354
  sim=0.76 → hybrid=0.6594


In [16]:
import shutil, os

SRC  = "../Progress/code/datasets/mapping"
DEST = "datasets"
os.makedirs(DEST, exist_ok=True)

files = [
    "occupation_gcsi.csv",
    "occupation_skill_bridge_onet.csv",
    "occupation_technology_skills_onet_summary.csv",
    "program_field_dropdown.csv",
    "program_recommendations.csv",
]

for f in files:
    shutil.copy2(f"{SRC}/{f}", f"{DEST}/{f}")
    size = os.path.getsize(f"{DEST}/{f}") / (1024*1024)
    print(f"✓ {f} ({size:.1f} MB)")

print(f"\nAll files in Final/datasets/:")
for f in sorted(os.listdir(DEST)):
    size = os.path.getsize(f"{DEST}/{f}") / (1024*1024)
    print(f"  {f} ({size:.1f} MB)")

✓ occupation_gcsi.csv (0.1 MB)
✓ occupation_skill_bridge_onet.csv (12.6 MB)
✓ occupation_technology_skills_onet_summary.csv (0.1 MB)
✓ program_field_dropdown.csv (2.4 MB)
✓ program_recommendations.csv (103.9 MB)

All files in Final/datasets/:
  occupation_gcsi.csv (0.1 MB)
  occupation_skill_bridge_onet.csv (12.6 MB)
  occupation_technology_skills_onet_summary.csv (0.1 MB)
  program_best_enriched_gcsi.csv (11.8 MB)
  program_field_dropdown.csv (2.4 MB)
  program_recommendations.csv (103.9 MB)
